In [0]:
%run "/Workspace/Users/sufianaslam127@gmail.com/audit_helper"

In [0]:
from datetime import datetime, timezone

run_start = datetime.now(timezone.utc)

print("Orders Silver audit run started.")

Orders Silver audit run started.


In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    lit,
    when,
    lower,
    trim,
    unix_timestamp
)
from pyspark.sql.types import DoubleType, IntegerType

bronze_path = "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/orders/"
silver_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/orders/"
quarantine_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/_quarantine/orders/"

[audit] logged run c46bcb24-e431-4f98-a94a-1eb65418f70e (success), latency=0.0s


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

bronze_count = bronze_df.count()

print("Bronze Orders:", bronze_count)

Bronze Orders: 60


In [0]:
typed_df = (
    bronze_df
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
)

In [0]:
normalized_currency = lower(trim(col("currency")))

typed_df = typed_df.withColumn(
    "currency",
    when(normalized_currency == "pkr", lit("PKR"))
    .when(normalized_currency == "pakistani rupee", lit("PKR"))
    .when(normalized_currency == "rs", lit("PKR"))
    .otherwise(col("currency")),
)

In [0]:
ACCEPTED_STATUSES = [
    "completed",
    "pending",
    "cancelled"
]

validated_df = typed_df.withColumn(
    "failure_reason",
    when(col("event_id").isNull(), lit("missing_event_id"))
    .when(col("order_id").isNull(), lit("missing_order_id"))
    .when(col("customer_id").isNull(), lit("missing_customer_id"))
    .when(
        col("amount").isNull() | (col("amount") < 0),
        lit("invalid_amount")
    )
    .when(
        col("quantity").isNull() | (col("quantity") < 1),
        lit("invalid_quantity")
    )
    .when(
        col("event_timestamp").isNull(),
        lit("unparseable_timestamp")
    )
    .when(
        ~col("status").isin(ACCEPTED_STATUSES),
        lit("unknown_status")
    )
    .otherwise(lit(None))
)

In [0]:
validated_df = validated_df.withColumn(
    "ingestion_delay_seconds",
    unix_timestamp(col("bronze_loaded_at"))
    - unix_timestamp(col("event_timestamp"))
)

LATE_THRESHOLD_SECONDS = 120

validated_df = validated_df.withColumn(
    "is_late",
    col("ingestion_delay_seconds") > LATE_THRESHOLD_SECONDS
)

In [0]:
pre_dedup_valid_df = validated_df.filter(
    col("failure_reason").isNull()
)

quarantine_df = validated_df.filter(
    col("failure_reason").isNotNull()
)

pre_dedup_valid_count = pre_dedup_valid_df.count()
quarantined_count = quarantine_df.count()

print("Pre-dedup valid:", pre_dedup_valid_count)
print("Quarantined:", quarantined_count)

Pre-dedup valid: 53
Quarantined: 7


In [0]:
final_valid_df = pre_dedup_valid_df.dropDuplicates(["event_id"])

final_valid_count = final_valid_df.count()

duplicates_removed_count = (
    pre_dedup_valid_count - final_valid_count
)

print("Final valid:", final_valid_count)
print("Duplicates removed:", duplicates_removed_count)

Final valid: 52
Duplicates removed: 1


In [0]:
(
    final_valid_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(silver_path)
)

print("Orders Silver write completed.")

Orders Silver write completed.


In [0]:
(
    quarantine_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(quarantine_path)
)

print("Orders quarantine write completed.")

Orders quarantine write completed.


In [0]:
reconciled_total = (
    final_valid_count
    + quarantined_count
    + duplicates_removed_count
)

assert bronze_count == reconciled_total, (
    f"Reconciliation failed: "
    f"bronze={bronze_count}, "
    f"valid+quarantined+duplicates_removed={reconciled_total}"
)

print("==============================================")
print("NEXPULSE — ORDERS RECONCILIATION")
print("==============================================")
print(f"Bronze events:          {bronze_count}")
print(f"Valid Silver events:    {final_valid_count}")
print(f"Quarantined events:     {quarantined_count}")
print(f"Duplicates removed:     {duplicates_removed_count}")
print(f"Reconciled:             {reconciled_total}")
print("----------------------------------------------")
print("PASSED: Orders reconciliation")

NEXPULSE — ORDERS RECONCILIATION
Bronze events:          60
Valid Silver events:    52
Quarantined events:     7
Duplicates removed:     1
Reconciled:             60
----------------------------------------------
PASSED: Orders reconciliation


In [0]:
print("Quarantine Failure Reasons")
print("=" * 50)

(quarantine_df
    .groupBy("failure_reason")
    .count()
    .orderBy("count", ascending=False)
    .show(truncate=False)
)

Quarantine Failure Reasons
+-------------------+-----+
|failure_reason     |count|
+-------------------+-----+
|invalid_amount     |4    |
|missing_customer_id|2    |
|unknown_status     |1    |
+-------------------+-----+



In [0]:
orders_df = spark.read.format("delta").load(
    "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/orders/"
)

print("Bronze Orders:", orders_df.count())

Bronze Orders: 60


In [0]:
try:
    log_pipeline_run(
        pipeline_name="silver_orders",
        source="orders",
        start_time=run_start,
        records_read=bronze_count,
        records_written=final_valid_count,
        records_quarantined=quarantined_count,
        records_deduplicated=duplicates_removed_count,
        status="success"
    )

except Exception as e:
    log_pipeline_run(
        pipeline_name="silver_orders",
        source="orders",
        start_time=run_start,
        status="failed",
        error_message=str(e)[:500]
    )
    raise

[audit] logged run 48d2f909-4217-4bad-ad68-d875654dfcdf (success), latency=25.3s


In [0]:
# ============================================================
# VERIFY AUDIT TABLE — LATEST RUNS
# ============================================================

from pyspark.sql.functions import col

audit_df = (
    spark.read
    .format("delta")
    .load(audit_table_path)
)

display(
    audit_df
    .orderBy(col("start_time").desc())
    .limit(5)
)

pipeline_run_id,pipeline_name,source,start_time,end_time,records_read,records_written,records_quarantined,records_deduplicated,status,error_message,processing_latency_seconds
48d2f909-4217-4bad-ad68-d875654dfcdf,silver_orders,orders,2026-08-22T13:57:41.842647Z,2026-08-22T13:58:07.137894Z,60,52,7,1,success,null,25.295247
c46bcb24-e431-4f98-a94a-1eb65418f70e,audit_helper_test,test,2026-08-22T13:57:22.498773Z,2026-08-22T13:57:22.498845Z,1,1,0,0,success,null,7.2E-5
cb24b51a-43ca-48aa-8379-89d3340c939f,audit_helper_test,test,2026-08-22T13:56:53.804145Z,2026-08-22T13:56:53.804197Z,1,1,0,0,success,null,5.2E-5
05a76169-5b03-452f-8fcc-a4542306f6b6,audit_helper_test,test,2026-08-22T13:49:42.089198Z,2026-08-22T13:49:42.089323Z,1,1,0,0,success,null,1.25E-4
c60d0f6b-bfd7-4d9e-aa3c-d735c17d74ca,audit_helper_test,test,2026-08-22T13:41:27.598137Z,2026-08-22T13:41:27.598214Z,1,1,0,0,success,null,7.7E-5


In [0]:
# ============================================================
# VERIFY ORDERS AUDIT RECONCILIATION
# ============================================================

latest_orders_audit = (
    audit_df
    .filter(col("pipeline_name") == "silver_orders")
    .orderBy(col("start_time").desc())
    .limit(1)
)

display(latest_orders_audit)

pipeline_run_id,pipeline_name,source,start_time,end_time,records_read,records_written,records_quarantined,records_deduplicated,status,error_message,processing_latency_seconds
48d2f909-4217-4bad-ad68-d875654dfcdf,silver_orders,orders,2026-08-22T13:57:41.842647Z,2026-08-22T13:58:07.137894Z,60,52,7,1,success,null,25.295247
